# Stacks and Queues: Restricting the Interface

**Learning goals**

1. See why *removing* operations from an ADT can be a feature.
2. Define the **Stack** and **Queue** ADTs as abstract base classes.
3. Read a linked-list implementation of each, then write a Python-list implementation of each against the *same* interface.
4. Discover, by measurement, that one of those list implementations has a hidden O(n) cost — and (challenge) fix it.

Run cells in order. Cells marked **✏️** are yours to complete.

## 1. Why drop `get(i)`?

Last notebook ended on this point: positional access is part of the List ADT, but it's the one thing a linked list is *bad* at. What a linked list is *good* at is touching the ends.

So here's an idea: what if we define ADTs that **only** allow touching the ends? We lose `get(i)`, `set(i)`, `insert(i)`, `delete(i)` — and in return every remaining operation can be O(1).

Two such ADTs come up constantly:

| ADT | Where you add | Where you remove | Nickname | Everyday example |
|---|---|---|---|---|
| **Stack** | top | top | LIFO — last in, first out | undo history, browser back button, the call stack |
| **Queue** | back | front | FIFO — first in, first out | print queue, customer line, task scheduler |

Neither lets you look in the middle. That restriction is the *point*: it's a promise to the user about behaviour, and it frees the implementer to pick whatever structure makes those few operations fast.

## 2. Setup: the `Node` class

Same `Node` as last time. We'll build the linked versions directly from nodes rather than wrapping `LinkedList` — a stack or queue only needs two or three operations, so dragging in the whole List ADT would be overkill.

In [ ]:
class Node:
    """A single element of a singly linked chain."""
    def __init__(self, data, next=None):
        self.data = data
        self.next = next

    def __repr__(self):
        return f"Node({self.data!r})"

## 3. The Stack ADT

Operations:

| Operation | Meaning |
|---|---|
| `push(value)` | put `value` on top |
| `pop()` | remove and return the top value (`IndexError` if empty) |
| `peek()` | return the top value without removing it (`IndexError` if empty) |
| `is_empty()` | `True` if no items |
| `len(stack)` | number of items |

We write the ADT down *as code* using Python's `abc` module. An abstract base class can't be instantiated, and any subclass that forgets to implement an `@abstractmethod` can't be either. It's the contract, enforced.

In [ ]:
from abc import ABC, abstractmethod

class Stack(ABC):
    """The Stack ADT. Subclasses must implement every method below."""

    @abstractmethod
    def push(self, value): ...

    @abstractmethod
    def pop(self): ...

    @abstractmethod
    def peek(self): ...

    @abstractmethod
    def is_empty(self): ...

    @abstractmethod
    def __len__(self): ...

    def __repr__(self):
        # Shared by all implementations: bottom -> top
        return f"{type(self).__name__}(bottom -> {list(self)} <- top)"


# You can't build the abstract thing itself:
try:
    Stack()
except TypeError as e:
    print("TypeError:", e)

### 3.1 `LinkedStack` — a worked implementation

Everything happens at the **head** of the chain, because that's the end a singly linked list can reach in O(1). "Top of the stack" = `self.head`.

```
push(3):      head -> 3 -> 2 -> 1 -> None     (new node becomes head)
pop():        head -> 2 -> 1 -> None          (head moves to head.next, returns 3)
```

Read every line — you'll write the list version yourself next.

In [ ]:
class LinkedStack(Stack):
    """Stack backed by a chain of Nodes; the head is the top."""

    def __init__(self):
        self.head = None
        self._size = 0

    def push(self, value):
        # New node points at the old top, then becomes the top.
        self.head = Node(value, self.head)
        self._size += 1

    def pop(self):
        if self.head is None:
            raise IndexError("pop from empty stack")
        removed = self.head
        self.head = removed.next       # unlink the top node
        self._size -= 1
        return removed.data

    def peek(self):
        if self.head is None:
            raise IndexError("peek at empty stack")
        return self.head.data

    def is_empty(self):
        return self.head is None

    def __len__(self):
        return self._size

    def __iter__(self):
        # Yield bottom -> top so __repr__ reads naturally.
        items = []
        current = self.head
        while current is not None:
            items.append(current.data)
            current = current.next
        return reversed(items)


s = LinkedStack()
for x in [1, 2, 3]:
    s.push(x)
print(s)
print("peek:", s.peek())
print("pop:", s.pop(), "| pop:", s.pop())
print(s, "| len:", len(s), "| empty?", s.is_empty())

### 3.2 ✏️ `ListStack` — your turn

Same interface, different structure: store the items in a Python list, `self._items`. Decide which *end* of the list is the top. Hint: think about which list operations are O(1) (look back at last notebook's cost table) and pick the end that makes **both** `push` and `pop` cheap.

Leave `__iter__` as written — it's only used for printing.

In [ ]:
class ListStack(Stack):
    """Stack backed by a Python list."""

    def __init__(self):
        self._items = []

    def push(self, value):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def pop(self):
        # ---- YOUR CODE HERE ----  (raise IndexError if empty)
        raise NotImplementedError

    def peek(self):
        # ---- YOUR CODE HERE ----  (raise IndexError if empty)
        raise NotImplementedError

    def is_empty(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def __len__(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def __iter__(self):
        # bottom -> top, assuming the END of the list is the top
        return iter(self._items)

In [ ]:
# scratch space — try your ListStack here
s = ListStack()


In [ ]:
#@title Run the Stack tests { display-mode: "form" }
!wget -q https://raw.githubusercontent.com/mdevlin-midpac/ib-cs-2027/main/data_structures/sq_tests.py -O sq_tests.py
from sq_tests import check_stack
check_stack(ListStack, name="ListStack")

### 3.3 Both are O(1) — so which is better?

For a stack, both implementations make every operation O(1). The Python-list version usually wins on raw speed (contiguous memory, no per-item object allocation) and is what you'd use in practice. The linked version wins if you need to guarantee *no* resize pause ever, or if nodes are already lying around.

The interesting story is the queue.

## 4. The Queue ADT

| Operation | Meaning |
|---|---|
| `enqueue(value)` | add `value` to the **back** |
| `dequeue()` | remove and return the **front** value (`IndexError` if empty) |
| `peek()` | return the front value without removing it |
| `is_empty()` / `len()` | as before |

In [ ]:
class Queue(ABC):
    """The Queue ADT."""

    @abstractmethod
    def enqueue(self, value): ...

    @abstractmethod
    def dequeue(self): ...

    @abstractmethod
    def peek(self): ...

    @abstractmethod
    def is_empty(self): ...

    @abstractmethod
    def __len__(self): ...

    def __repr__(self):
        return f"{type(self).__name__}(front -> {list(self)} <- back)"

### 4.1 `LinkedQueue` — a worked implementation

A queue touches **both** ends: remove at the front, add at the back. Removing at the head is O(1) as before. Adding at the back would mean walking the whole chain… unless we also keep a **`tail`** pointer to the last node. Then both ends are O(1).

```
enqueue(3):   head -> 1 -> 2 -> 3 -> None        tail.next = new; tail = new
                                 ^tail
dequeue():    head -> 2 -> 3 -> None             head = head.next, returns 1
                           ^tail
```

The price of a `tail` pointer is bookkeeping: two special cases where `head` and `tail` must be kept in sync — enqueuing into an **empty** queue, and dequeuing the **last** item. Watch for them below.

In [ ]:
class LinkedQueue(Queue):
    """Queue backed by a chain of Nodes with head (front) and tail (back) pointers."""

    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def enqueue(self, value):
        new_node = Node(value)
        if self.tail is None:
            # Special case: queue was empty -> the new node is both front and back.
            self.head = new_node
        else:
            self.tail.next = new_node   # hang it off the old back...
        self.tail = new_node            # ...and it becomes the new back.
        self._size += 1

    def dequeue(self):
        if self.head is None:
            raise IndexError("dequeue from empty queue")
        removed = self.head
        self.head = removed.next
        if self.head is None:
            # Special case: that was the last node -> tail must be cleared too.
            self.tail = None
        self._size -= 1
        return removed.data

    def peek(self):
        if self.head is None:
            raise IndexError("peek at empty queue")
        return self.head.data

    def is_empty(self):
        return self.head is None

    def __len__(self):
        return self._size

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.data
            current = current.next


q = LinkedQueue()
for x in ["a", "b", "c"]:
    q.enqueue(x)
print(q)
print("dequeue:", q.dequeue(), "| peek:", q.peek())
print(q, "| len:", len(q))

**Check your understanding:** in `dequeue`, what would go wrong if we forgot the `self.tail = None` line, then called `enqueue` again? Trace it by hand.

### 4.2 ✏️ `ListQueue` — your turn

Again: same interface, but store the items in a Python list `self._items`. Write the most direct implementation you can think of — don't optimise yet. Front of the queue = front of the list.

In [ ]:
class ListQueue(Queue):
    """Queue backed by a Python list. Front of the queue = index 0."""

    def __init__(self):
        self._items = []

    def enqueue(self, value):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def dequeue(self):
        # ---- YOUR CODE HERE ----  (raise IndexError if empty)
        raise NotImplementedError

    def peek(self):
        # ---- YOUR CODE HERE ----  (raise IndexError if empty)
        raise NotImplementedError

    def is_empty(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def __len__(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def __iter__(self):
        return iter(self._items)

In [ ]:
# scratch space — try your ListQueue here
q = ListQueue()


In [ ]:
#@title Run the Queue tests { display-mode: "form" }
!wget -q https://raw.githubusercontent.com/mdevlin-midpac/ib-cs-2027/main/data_structures/sq_tests.py -O sq_tests.py
from sq_tests import check_queue
check_queue(ListQueue, name="ListQueue")

## 5. Same interface, very different cost

Both queues pass the same tests. Now let's time them: enqueue N items, then dequeue all N.

In [ ]:
import time

def time_queue(QueueClass, n):
    q = QueueClass()
    t0 = time.perf_counter()
    for i in range(n):
        q.enqueue(i)
    for _ in range(n):
        q.dequeue()
    return time.perf_counter() - t0

for n in [5_000, 20_000, 80_000]:
    t_linked = time_queue(LinkedQueue, n)
    t_list   = time_queue(ListQueue, n)
    print(f"n={n:>6}   LinkedQueue {t_linked:7.3f}s   ListQueue {t_list:7.3f}s   ratio {t_list / t_linked:5.1f}x")

If your `ListQueue` uses `self._items.pop(0)` for `dequeue`, you should see the ratio *grow* as `n` grows — the list version is getting worse faster than linearly.

**Why?** Python's list is a dynamic array. `pop(0)` removes the first slot, and every remaining element has to shift left one position: O(n) per dequeue, O(n²) to drain the queue. `LinkedQueue.dequeue` just moves the `head` pointer: O(1).

This is the lesson of the whole unit in one table: the ADT is identical, the tests are identical, the user can't tell the difference — until it's slow.

**What Python actually ships:** `collections.deque` ("double-ended queue"). It's implemented as a linked list of small fixed-size blocks, giving O(1) at both ends with good constant factors. In real code, `deque` is what you reach for:

In [ ]:
from collections import deque

def time_deque(n):
    d = deque()
    t0 = time.perf_counter()
    for i in range(n):
        d.append(i)
    for _ in range(n):
        d.popleft()
    return time.perf_counter() - t0

for n in [5_000, 20_000, 80_000]:
    print(f"n={n:>6}   deque {time_deque(n):7.3f}s")

## 6. ✏️ Exercises

### Exercise 1 — Balanced brackets

Write `is_balanced(text)` that returns `True` if every `(`, `[`, `{` is closed by the matching `)`, `]`, `}` in the right order. Use a stack (either implementation — that's the point of the ADT).

```
is_balanced("([]{})")   -> True
is_balanced("([)]")     -> False
is_balanced("((")       -> False
```

In [ ]:
def is_balanced(text):
    # Your code here
    raise NotImplementedError

# quick checks
for s, expected in [("([]{})", True), ("([)]", False), ("((", False), ("", True), ("a(b)c", True)]:
    try:
        print(f"{s!r:12} -> {is_balanced(s)}   (expected {expected})")
    except NotImplementedError:
        print("not implemented yet"); break

### Exercise 2 — Round-robin scheduler

You have a list of `(task_name, time_needed)` pairs and a time slice `quantum`. Simulate a round-robin scheduler: each task runs for up to `quantum` units, then (if unfinished) goes to the **back** of the queue. Return the order in which tasks *finish*.

```
round_robin([("A", 5), ("B", 2), ("C", 3)], quantum=2)  ->  ["B", "C", "A"]
```
Use a queue. Think about which implementation you'd want if there were a million tasks.

In [ ]:
def round_robin(tasks, quantum):
    # Your code here
    raise NotImplementedError

try:
    print(round_robin([("A", 5), ("B", 2), ("C", 3)], quantum=2))   # expected ['B', 'C', 'A']
except NotImplementedError:
    print("not implemented yet")

### Exercise 3 — Stack from two queues (or queue from two stacks)

Pick one:
* `TwoStackQueue(Queue)`: implement a queue using only two `ListStack`s internally.
* `TwoQueueStack(Stack)`: implement a stack using only two `LinkedQueue`s internally.

Run it against the matching test function (`check_queue` or `check_stack`) — because it satisfies the ADT, the tests don't care what's inside. What's the cost of each operation?

In [ ]:
# Your code here


### 🏆 CHALLENGE — `FastListQueue`: an O(1) queue on a Python list

`ListQueue` was slow because `pop(0)` shifts everything. Can you keep a Python list as the storage but make `dequeue` O(1)?

**The trick:** don't remove from the front at all. Keep an index `self._front` that points at the current front item. `dequeue` returns `self._items[self._front]` and then does `self._front += 1`. The "dead" slots stay behind.

**Two problems to solve:**
1. `len`, `is_empty`, `peek`, and `__iter__` must account for the dead slots.
2. Dead slots leak memory forever. Fix: whenever `self._front` gets large (say, more than half the list), compact — slice off the dead prefix and reset `_front` to 0. That compaction is O(n), but it happens rarely, so the *amortized* cost per dequeue is still O(1).

The tests check correctness, that dequeue time scales linearly (not quadratically), and that the dead slots actually get reclaimed. Compare your timings against `deque` when you're done.

In [ ]:
class FastListQueue(Queue):
    """Queue on a Python list with O(1) amortized dequeue."""

    def __init__(self):
        self._items = []
        self._front = 0      # index of the current front item

    def enqueue(self, value):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def dequeue(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def peek(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def is_empty(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def __len__(self):
        # ---- YOUR CODE HERE ----
        raise NotImplementedError

    def __iter__(self):
        return iter(self._items[self._front:])

In [ ]:
#@title Run the CHALLENGE tests { display-mode: "form" }
!wget -q https://raw.githubusercontent.com/mdevlin-midpac/ib-cs-2027/main/data_structures/sq_tests.py -O sq_tests.py
from sq_tests import check_fast_queue
check_fast_queue(FastListQueue)

### Exercise 4 — Think about it (no code)

1. Why doesn't `LinkedStack` need a `tail` pointer, while `LinkedQueue` does?
2. `ListStack` is O(1) for everything and `ListQueue` isn't. What single property of dynamic arrays explains the difference?
3. A **deque** supports add/remove at *both* ends. Sketch the interface. Could you implement it on a *singly* linked list with O(1) for all four operations? What would you need to change about `Node`?
4. A **priority queue** dequeues the *smallest* (or highest-priority) item rather than the oldest. Write its interface as an ABC. Which of our structures — linked list, sorted linked list, Python list — would you use to back it, and what would each operation cost?

---
## Summary

* **Stacks** (LIFO) and **queues** (FIFO) are ADTs that deliberately *drop* positional access, keeping only end operations.
* Writing the ADT as an `ABC` makes the contract explicit and enforced; any class satisfying it passes the same tests.
* A linked list backs a stack with `head` only, and a queue with `head` + `tail` — all O(1).
* A Python list backs a stack perfectly, but a naive list queue is O(n) per dequeue because `pop(0)` shifts the array. A front-index with periodic compaction fixes it; `collections.deque` is the production answer.
* Same interface, different structure, different cost — and the only way to know is to measure.